# Clase 6 — Órdenes y matching

Enviar órdenes contra el libro y ver cómo se cruzan. Market, limit, IOC y FOK: cada tipo cambia el coste, la probabilidad de ejecución y el riesgo.

**Hoy construyes:** MatchingEngine: cómo se cruzan las órdenes.

## Cómo usar este cuaderno

- **Núcleo (en clase):** ejercicios 1 a 3.
- **Si vamos bien:** ejercicios 4 en adelante.
- **Casa / auxiliares:** el cuaderno `*_auxiliary.ipynb`.

Inténtalo, ejecuta la comprobación (`assert`) y mira la solución solo si te atascas.

## 1. Una market order se llena

**Practicas:** MatchingEngine + MARKET.

Cruza una market buy de 0.5 contra el primer snapshot. Guarda `fills` y `filled` (suma de tamaños).

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
fills = None
filled = None

In [ ]:
assert abs(filled - 0.5) < 1e-9, 'una market siempre se llena si hay liquidez'
assert all(f.side == Side.BUY for f in fills)
print('ok  fills=%d' % len(fills))

### Solución guiada

```python
order = Order('BTCUSDT', Side.BUY, 0.5, order_type=OrderType.MARKET)
fills = eng.process(order, book)
filled = sum(f.size for f in fills)
```

## 2. Una limit cruza solo a su precio

**Practicas:** LIMIT + remanente.

Envía una limit buy enorme (size=999) al `best_bid` actual. No debería cruzar nada (precio por debajo del ask). Guarda `n_fills`.

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
bb = book.best_bid
n_fills = None

In [ ]:
assert n_fills == 0, 'comprar al best_bid no cruza el ask'
print('ok')

### Solución guiada

```python
order = Order('BTCUSDT', Side.BUY, 999, price=bb, order_type=OrderType.LIMIT)
fills = eng.process(order, book)
n_fills = len(fills)
```

## 3. Una market crossing limit sí cruza

**Practicas:** limit marketable.

Envía una limit buy de 0.3 a un precio por encima del `best_ask` (best_ask + 100). Cruza. Guarda `filled`.

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
ba = book.best_ask
filled = None

In [ ]:
assert abs(filled - 0.3) < 1e-9
print('ok')

### Solución guiada

```python
order = Order('BTCUSDT', Side.BUY, 0.3, price=ba+100, order_type=OrderType.LIMIT)
fills = eng.process(order, book)
filled = sum(f.size for f in fills)
```

## 4. FOK: todo o nada

**Practicas:** OrderType.FOK.

Envía una FOK buy de tamaño 9999 (más de lo que hay) a precio muy alto. Debe devolver 0 fills. Guarda `n_fills`.

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
eng = MatchingEngine()
ba = book.best_ask
n_fills = None

In [ ]:
assert n_fills == 0, 'FOK no se llena entera -> 0 fills'
print('ok')

### Solución guiada

```python
order = Order('BTCUSDT', Side.BUY, 9999, price=ba+1000, order_type=OrderType.FOK)
fills = eng.process(order, book)
n_fills = len(fills)
```

## 5. Precio efectivo de una market

**Practicas:** vwap de los fills.

Cruza una market buy de 1.0 y calcula `eff_price` = nocional total / tamaño total. Debe ser >= best_ask (pagas el barrido).

In [ ]:
from exchange import Market, MatchingEngine, Order, Side, OrderType
book = Market.sample().step()
ba = book.best_ask
eng = MatchingEngine()
eff_price = None

In [ ]:
assert eff_price >= ba - 1e-6, 'una market barre niveles: precio efectivo >= best_ask'
print('ok  eff=%.2f best_ask=%.2f' % (eff_price, ba))

### Solución guiada

```python
order = Order('BTCUSDT', Side.BUY, 1.0, order_type=OrderType.MARKET)
fills = eng.process(order, book)
eff_price = sum(f.price*f.size for f in fills) / sum(f.size for f in fills)
```

## Cierre

La forma en que envías la orden decide tu coste: cruzar ya, o esperar barato y arriesgarte a no ejecutar.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.